# Synthetic Benchmark — LLM Correctness Evaluation Report

**Purpose:** Analyze the quality/correctness of synthetic benchmark task definitions, *not agent performance*. The same four dimensions were judged separately: **task consistency (T), expected outcome validity (O), policy compliance (P), and workflow feasibility (F)**.

Each dimension receives **1.0** (supported), **0.5** (uncertain/insufficient evidence), or **0.0** (established violation). A task score is the equal-weight mean of its four dimensions; the **overall score is the task-weighted mean across all evaluated tasks**. A score is a judge assessment, not independently verified ground truth.

Run cells top-to-bottom. The notebook reads seven subfolders of `llm_synthetic_results`. It can also use `llm_synthetic_results.zip` if no extracted folder is present. Data is never modified.

## 0. Setup and paths

Required packages: `pandas`, `numpy`, `matplotlib`, `seaborn`, and `IPython`. Optional: `ipywidgets` for interactive domain/task selectors. Install missing dependencies in your notebook environment, for example `pip install pandas numpy matplotlib seaborn ipywidgets`.

Set `RESULTS_DIR` if your notebook lives outside the project root. `judge_details.json` is the authoritative per-task source for scores **and** explanations. CSV reports are loaded for consistency checks.

In [1]:
from pathlib import Path
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 160)

# Change this if needed, e.g. Path("../llm_synthetic_results")
RESULTS_DIR = Path("llm_synthetic_results")
EXPECTED_DOMAINS = 7
EXPECTED_TASKS = 64  # Completeness check only; never used to calculate averages.

# Discover folder relative to notebook directory / project root.
candidates = [RESULTS_DIR, Path("../llm_synthetic_results"), Path("../../llm_synthetic_results")]
found = next((p for p in candidates if p.is_dir()), None)

# Optional ZIP fallback; do not overwrite an existing extracted results directory.
if found is None:
    archives = [Path("llm_synthetic_results.zip"), Path("../llm_synthetic_results.zip")]
    archive = next((p for p in archives if p.is_file()), None)
    if archive is not None:
        extract_parent = Path.cwd() / "_synthetic_results_readonly_extract"
        with zipfile.ZipFile(archive) as zf:
            safe_members = [m for m in zf.infolist()
                            if not m.filename.startswith("__MACOSX/")
                            and not Path(m.filename).is_absolute()
                            and ".." not in Path(m.filename).parts]
            zf.extractall(extract_parent, members=safe_members)
        found = extract_parent / "llm_synthetic_results"

if found is None or not found.is_dir():
    raise FileNotFoundError("Cannot find llm_synthetic_results/. Set RESULTS_DIR to your project folder.")
RESULTS_DIR = found.resolve()
print("Reading results from:", RESULTS_DIR)


ModuleNotFoundError: No module named 'matplotlib'

## 1. Load and validate the seven dataset results

Each domain folder is expected to contain `judge_details.json`, `task_scores.csv`, and `domain_scores.csv`. The domain's `OVERALL` CSV row is a summary **for that folder only**; it must **not** be treated as another task or averaged across folders to obtain the full benchmark score.

In [ ]:
DIMENSIONS = ["task_consistency", "expected_outcome_validity", "policy_compliance", "workflow_feasibility"]
SHORT = {"task_consistency": "Task consistency", "expected_outcome_validity": "Outcome validity",
         "policy_compliance": "Policy compliance", "workflow_feasibility": "Workflow feasibility"}
DOMAIN_NAMES = {
    "banking_customer_onboarding": "Banking customer onboarding",
    "banking_accounts_deposits": "Banking accounts & deposits",
    "banking_fraud_monitoring": "Banking fraud monitoring",
    "healthcare_patient_onboarding": "Healthcare patient onboarding",
    "healthcare_prior_authorization": "Healthcare prior authorization",
    "insurance_claims_intake": "Insurance claims intake",
    "manufacturing_purchase_to_pay": "Manufacturing purchase-to-pay",
}

records, csv_task_frames, csv_domain_frames = [], [], []
folders = sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
for folder in folders:
    required = [folder / "judge_details.json", folder / "task_scores.csv", folder / "domain_scores.csv"]
    if not all(p.is_file() for p in required):
        warnings.warn(f"Skipping {folder.name}: expected result files not all present")
        continue
    details = json.loads(required[0].read_text(encoding="utf-8"))
    if not isinstance(details, list):
        raise ValueError(f"Expected a list in {required[0]}")
    for r in details:
        if not all(dim in r.get("dimensions", {}) for dim in DIMENSIONS):
            raise ValueError(f"Missing dimension in {folder.name} task {r.get('task_id')}")
        records.append(r)
    task_csv = pd.read_csv(required[1], dtype={"task_id": str, "domain": str})
    task_csv["source_folder"] = folder.name
    csv_task_frames.append(task_csv)
    domain_csv = pd.read_csv(required[2])
    domain_csv["source_folder"] = folder.name
    csv_domain_frames.append(domain_csv)

if not records:
    raise ValueError("No valid judgment records found in the results folder")

csv_tasks = pd.concat(csv_task_frames, ignore_index=True)
csv_domains = pd.concat(csv_domain_frames, ignore_index=True)

# One row per task: scores plus descriptive metadata from JSON.
task_rows = []
judgment_rows = []
for r in records:
    task_row = {"domain": r["domain"], "task_id": str(r["task_id"]),
                "title": r.get("title") or "", "task_score": float(r["task_score"]),
                "model": r.get("model"), "prompt_version": r.get("prompt_version")}
    for dim in DIMENSIONS:
        j = r["dimensions"][dim]
        task_row[dim] = float(j["score"])
        judgment_rows.append({"domain": r["domain"], "task_id": str(r["task_id"]),
                              "title": r.get("title") or "", "task_score": float(r["task_score"]),
                              "dimension": dim, "dimension_label": SHORT[dim],
                              "score": float(j["score"]),
                              "explanation": j.get("explanation", ""),
                              "evidence": j.get("evidence", [])})
    task_rows.append(task_row)

tasks = pd.DataFrame(task_rows)
judgments = pd.DataFrame(judgment_rows)
tasks["domain_label"] = tasks["domain"].map(DOMAIN_NAMES).fillna(tasks["domain"])
judgments["domain_label"] = judgments["domain"].map(DOMAIN_NAMES).fillna(judgments["domain"])

if tasks.duplicated(["domain", "task_id"]).any():
    raise ValueError("Duplicate (domain, task_id) found in judge_details.json")
if not np.allclose(tasks["task_score"], tasks[DIMENSIONS].mean(axis=1)):
    raise ValueError("Some task scores do not match the mean of their four dimensions")
if not judgments["score"].isin([0.0, 0.5, 1.0]).all():
    raise ValueError("Unexpected dimension scores outside {0.0, 0.5, 1.0}")

# Cross-check the flattened CSV against the JSON records.
joined = tasks.merge(csv_tasks, on=["domain", "task_id"], how="outer", suffixes=("_json", "_csv"), indicator=True)
if not (joined["_merge"] == "both").all():
    warnings.warn("JSON and task_scores.csv have different task IDs; using JSON as authoritative")
else:
    for name in ["task_score", *DIMENSIONS]:
        if not np.allclose(joined[f"{name}_json"], joined[f"{name}_csv"], equal_nan=False):
            warnings.warn(f"Discrepancy between JSON and CSV for {name}; using JSON")

num_domains = tasks["domain"].nunique()
num_tasks = len(tasks)
print(f"Loaded {num_tasks} tasks from {num_domains} domains; {len(judgments)} dimension judgments.")
if num_domains != EXPECTED_DOMAINS or num_tasks != EXPECTED_TASKS:
    warnings.warn(f"Expected {EXPECTED_TASKS} tasks in {EXPECTED_DOMAINS} domains; found {num_tasks} in {num_domains}. "
                  "Scores still use the actual loaded records. Verify updated input files if needed.")
display(tasks.groupby("domain_label", as_index=False).agg(tasks=("task_id", "count"), mean_correctness=("task_score", "mean")).sort_values("domain_label"))


## 2. Dataset overview — task-weighted correctness

**Interpretation:** The dataset-wide correctness score is the mean of **individual task scores**, not the unweighted mean of seven domain averages. Higher means more of the provided task definitions/outcomes were supported by the judge's evidence. The uncertainty and violation counts refer to **dimension judgments**, not numbers of faulty tasks.

In [ ]:
overall = tasks["task_score"].mean()
counts = judgments["score"].value_counts().reindex([1.0, 0.5, 0.0], fill_value=0)
uncertain_n = int(counts.loc[0.5]); violated_n = int(counts.loc[0.0]); supported_n = int(counts.loc[1.0])

fig, axes = plt.subplots(1, 4, figsize=(15, 2.8))
for ax, (value, label) in zip(axes, [
    (f"{overall:.3f}", "Overall correctness (0–1)"),
    (f"{num_tasks}", "Evaluated tasks"),
    (f"{uncertain_n}", "Uncertain judgments (0.5)"),
    (f"{violated_n}", "Violated judgments (0.0)")
]):
    ax.text(0.5, 0.58, value, ha="center", va="center", fontsize=27, weight="bold")
    ax.text(0.5, 0.18, label, ha="center", va="center", fontsize=10, wrap=True)
    ax.axis("off")
plt.suptitle("Synthetic dataset — correctness summary", fontsize=15)
plt.tight_layout()
plt.show()

print(f"Interpretation: Mean task correctness is {overall:.3f}/1.000 across {num_tasks} tasks. "
      f"Of {len(judgments)} dimension judgments, {supported_n} were supported, "
      f"{uncertain_n} uncertain, and {violated_n} established violations.")


## 3. Overall dimension averages and score distributions

**Interpretation:** The four-dimension bar chart pinpoints *which aspect of benchmark quality* is strongest or weakest. The histogram shows whether the overall average masks substantial variation between tasks. The count plot distinguishes **uncertain evidence (0.5)** from **established violations (0.0)**; they must not be combined into a single error count.

In [ ]:
dimension_means = tasks[DIMENSIONS].mean().rename(index=SHORT).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(dimension_means.index, dimension_means.values)
ax.set(xlim=(0, 1), xlabel="Average dimension score (0–1)", title="Average correctness by dimension")
for i, val in enumerate(dimension_means.values): ax.text(min(val + 0.015, .95), i, f"{val:.3f}", va="center")
plt.tight_layout(); plt.show()
print(f"Lowest-rated dimension: {dimension_means.index[0]} ({dimension_means.iloc[0]:.3f}); "
      f"highest-rated: {dimension_means.index[-1]} ({dimension_means.iloc[-1]:.3f}). "
      "These reflect the judge's evidence-based assessments, not independently established quality rankings.")

# With four increments of 0.5, task averages lie on a 0.125-spaced grid.
possible = np.round(np.arange(0, 1.0001, .125), 3)
score_freq = tasks["task_score"].round(3).value_counts().reindex(possible, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.bar([f"{x:.3f}" for x in score_freq.index], score_freq.values)
ax.set(xlabel="Overall task score", ylabel="Number of tasks", title="Distribution of task correctness scores")
plt.xticks(rotation=40)
plt.tight_layout(); plt.show()
print("Interpretation: The distribution shows whether tasks cluster near high support, uncertainty, or low correctness. "
      "A score of 0.5 is a task average and does not necessarily mean all dimensions were uncertain.")

fig, ax = plt.subplots(figsize=(7, 4.2))
labels = ["Supported (1.0)", "Uncertain (0.5)", "Violation (0.0)"]
vals = [supported_n, uncertain_n, violated_n]
ax.bar(labels, vals)
for i, val in enumerate(vals): ax.text(i, val + max(vals)*.015, str(val), ha="center")
ax.set(ylabel="Dimension judgments", title="Evidence status of all dimension judgments")
plt.tight_layout(); plt.show()
print(f"Interpretation: {uncertain_n}/{len(judgments)} ({uncertain_n/len(judgments):.1%}) judgments were uncertain; "
      f"{violated_n}/{len(judgments)} ({violated_n/len(judgments):.1%}) showed established violations.")


## 4. Domain-level average correctness

**Interpretation:** This chart compares the seven domain averages. Domain sample sizes are displayed because an average over six tasks should not be treated as equally precise as one over thirteen tasks. The overall dataset mean continues to weight every task equally.

In [ ]:
domain_stats = (tasks.groupby(["domain", "domain_label"], as_index=False)
                .agg(task_count=("task_id", "count"), mean_correctness=("task_score", "mean"))
                .sort_values("mean_correctness"))
fig, ax = plt.subplots(figsize=(10, 5.4))
ax.barh(domain_stats["domain_label"], domain_stats["mean_correctness"])
for i, r in enumerate(domain_stats.itertuples()):
    ax.text(min(r.mean_correctness + .012, .94), i, f"{r.mean_correctness:.3f} (n={r.task_count})", va="center", fontsize=9)
ax.axvline(overall, linestyle="--", color="black", alpha=.65, label=f"Task-weighted overall = {overall:.3f}")
ax.set(xlim=(0, 1), xlabel="Average task correctness (0–1)", title="Correctness by synthetic domain")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"Highest average domain: {domain_stats.iloc[-1]['domain_label']} ({domain_stats.iloc[-1]['mean_correctness']:.3f}). "
      f"Lowest: {domain_stats.iloc[0]['domain_label']} ({domain_stats.iloc[0]['mean_correctness']:.3f}). "
      "Different task complexity and evidence coverage may contribute to differences.")


## 5. Domain × dimension heatmap

**Interpretation:** Each cell averages the selected dimension across tasks in that domain. A relatively low value identifies **where** a domain receives weaker support or more uncertainty/violations (rather than merely having a low overall score).

In [ ]:
domain_dimension = tasks.groupby("domain_label")[DIMENSIONS].mean().rename(columns=SHORT)
order = domain_stats["domain_label"].tolist()
domain_dimension = domain_dimension.reindex(order)
fig, ax = plt.subplots(figsize=(11, 5.3))
sns.heatmap(domain_dimension, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="YlGnBu", ax=ax, cbar_kws={"label": "Mean correctness"})
ax.set(xlabel="Judge dimension", ylabel="", title="Domain-level correctness across four dimensions")
plt.xticks(rotation=18, ha="right"); plt.tight_layout(); plt.show()
print("Interpretation: Compare each domain's rows to distinguish policy, feasibility, outcome, and consistency weaknesses. "
      "Scores of 0.5 may reflect insufficient evidence, not confirmed incorrectness.")


## 6. Task-level score variation by domain

**Interpretation:** The boxplot summarizes spread; overlaid points show every actual task. A domain with a high mean but several low-scoring tasks may warrant targeted review. Domain counts are unequal.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.boxplot(data=tasks, x="domain_label", y="task_score", order=order, ax=ax, color="lightgray")
sns.stripplot(data=tasks, x="domain_label", y="task_score", order=order, ax=ax, jitter=.18, size=5)
ax.set(ylim=(-.04, 1.04), xlabel="", ylabel="Task correctness score", title="Distribution of task scores within each domain")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()
print("Interpretation: Points represent task-level averages; the spread shows consistency of judge ratings within each domain.")


## 7. Individual task scores and task × dimension heatmap

Select a domain below to display its individual task scores and four-dimension matrix. If widgets are unavailable, change `SELECTED_DOMAIN` and rerun the static fallback cell. Short task IDs are shown on chart axes; full titles are available in the next section.

In [ ]:
def plot_domain_tasks(domain):
    frame = tasks.loc[tasks["domain"] == domain].copy().sort_values(["task_score", "task_id"])
    if frame.empty:
        print("No tasks found for domain:", domain); return
    label = DOMAIN_NAMES.get(domain, domain)
    fig, ax = plt.subplots(figsize=(10, max(4.2, .4*len(frame)+1)))
    ax.barh(frame["task_id"], frame["task_score"])
    ax.set(xlim=(0, 1), xlabel="Mean of four dimension scores", title=f"Individual task scores — {label}")
    for i, score in enumerate(frame["task_score"]):
        ax.text(min(score + .01, .95), i, f"{score:.3f}", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

    matrix = frame.set_index("task_id")[DIMENSIONS].rename(columns=SHORT)
    fig, ax = plt.subplots(figsize=(9, max(3.5, .43*len(matrix)+1)))
    sns.heatmap(matrix, vmin=0, vmax=1, cmap="YlGnBu", annot=True, fmt=".1f", ax=ax,
                cbar_kws={"label": "Dimension score"})
    ax.set(title=f"Task × dimension judgments — {label}", xlabel="", ylabel="")
    plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()
    print("Interpretation: Each task average can conceal different combinations of supported, uncertain, and violated dimensions. "
          "Use the heatmap and explanations together before drawing conclusions about an individual task.")

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    domain_picker = widgets.Dropdown(options=[(DOMAIN_NAMES.get(d, d), d) for d in sorted(tasks["domain"].unique())],
                                     description="Domain:", layout=widgets.Layout(width="440px"))
    output = widgets.Output()
    def _update_domain(change=None):
        with output:
            clear_output(wait=True)
            plot_domain_tasks(domain_picker.value)
    domain_picker.observe(_update_domain, names="value")
    display(domain_picker, output)
    _update_domain()
except ImportError:
    print("ipywidgets is not installed. Run the static fallback below.")


In [ ]:
# Static fallback (also useful for exported, non-interactive notebooks).
# Uncomment to render for a chosen domain without widgets:
# SELECTED_DOMAIN = sorted(tasks["domain"].unique())[0]
# plot_domain_tasks(SELECTED_DOMAIN)


## 8. LLM judgment explorer — explanations and evidence

**Interpretation:** Read the explanation and cited evidence for each dimension before treating a low score as an actual quality defect. **0.0** means the judge identified an explicit violation; **0.5** means evidence was ambiguous or insufficient. Evidence strings are the judge's citations to supplied task/tool/policy material, not independently verified external citations.

In [ ]:
def show_task_explanations(domain, task_id):
    subset = judgments[(judgments["domain"] == domain) & (judgments["task_id"] == str(task_id))]
    if subset.empty:
        print("No matching task"); return
    head = subset.iloc[0]
    display(Markdown(f"### {head['title']}\n\n**Task ID:** `{task_id}` · **Overall correctness:** {head['task_score']:.3f}/1.000"))
    for dim in DIMENSIONS:
        j = subset[subset["dimension"] == dim].iloc[0]
        explanation = str(j["explanation"])
        evidence = j["evidence"] if isinstance(j["evidence"], list) else []
        print(f"\n{SHORT[dim]} — {j['score']:.1f}")
        print("Explanation:", explanation)
        if evidence:
            print("Evidence:")
            for e in evidence: print("  •", e)
        else:
            print("Evidence: none supplied")

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    exp_domain = widgets.Dropdown(options=[(DOMAIN_NAMES.get(d, d), d) for d in sorted(tasks["domain"].unique())],
                                  description="Domain:", layout=widgets.Layout(width="460px"))
    exp_task = widgets.Dropdown(description="Task:", layout=widgets.Layout(width="850px"))
    exp_output = widgets.Output()
    def _choose_tasks(change=None):
        avail = tasks[tasks["domain"] == exp_domain.value].sort_values("task_id")
        exp_task.options = [(f"{r.task_id} — {r.title[:75]}", r.task_id) for r in avail.itertuples()]
    def _show_explanations(change=None):
        with exp_output:
            clear_output(wait=True)
            if exp_task.value is not None:
                show_task_explanations(exp_domain.value, exp_task.value)
    exp_domain.observe(_choose_tasks, names="value")
    exp_task.observe(_show_explanations, names="value")
    _choose_tasks()
    display(widgets.VBox([exp_domain, exp_task]), exp_output)
    _show_explanations()
except ImportError:
    print("ipywidgets is not installed; use show_task_explanations(domain, task_id) directly.")


In [ ]:
# Static fallback: uncomment and edit the domain/task ID to inspect one task.
# CHOSEN_DOMAIN = sorted(tasks["domain"].unique())[0]
# CHOSEN_TASK = tasks.loc[tasks["domain"] == CHOSEN_DOMAIN, "task_id"].iloc[0]
# show_task_explanations(CHOSEN_DOMAIN, CHOSEN_TASK)


## 9. Review uncertain and violated judgments

This table makes it easier to locate the most important **potential quality issues**. It can be filtered to uncertain, violated, or both. Because explanations are generated judgments, the results indicate where **manual verification** may be useful; a 0.5 score does not establish an error.

In [ ]:
def issue_table(selection="Both", domain="All"):
    selected = judgments.copy()
    if selection == "Violations (0.0)": selected = selected[selected["score"] == 0.0]
    elif selection == "Uncertain (0.5)": selected = selected[selected["score"] == 0.5]
    else: selected = selected[selected["score"].isin([0.0, 0.5])]
    if domain != "All": selected = selected[selected["domain"] == domain]
    selected = selected.sort_values(["score", "domain_label", "task_id", "dimension"])
    view = selected[["domain_label", "task_id", "dimension_label", "score", "explanation", "evidence"]].copy()
    view["evidence"] = view["evidence"].apply(lambda x: " | ".join(map(str, x)) if isinstance(x, list) else str(x))
    display(view.rename(columns={"domain_label": "Domain", "task_id": "Task ID", "dimension_label": "Dimension",
                                 "score": "Score", "explanation": "LLM explanation", "evidence": "Supporting evidence"}))
    print(f"Showing {len(view)} judgments ({selection.lower()}); these are dimension counts, not distinct tasks.")

try:
    import ipywidgets as widgets
    issue_filter = widgets.Dropdown(options=["Both", "Violations (0.0)", "Uncertain (0.5)"], description="Show:")
    issue_domain = widgets.Dropdown(options=[("All domains", "All")] +
                                    [(DOMAIN_NAMES.get(d, d), d) for d in sorted(tasks["domain"].unique())],
                                    description="Domain:", layout=widgets.Layout(width="430px"))
    from IPython.display import clear_output
    issue_output = widgets.Output()
    def _update_issues(change=None):
        with issue_output:
            clear_output(wait=True)
            issue_table(issue_filter.value, issue_domain.value)
    issue_filter.observe(_update_issues, names="value")
    issue_domain.observe(_update_issues, names="value")
    display(widgets.HBox([issue_filter, issue_domain]), issue_output)
    _update_issues()
except ImportError:
    issue_table("Both", "All")


## 10. Key findings (automatically generated, descriptive)

These statements summarize the **observed judge outputs** and should be used alongside the detailed evidence. Differences between domains may reflect different task complexity, policies, or availability of reference context—not necessarily a definitive ranking of domain quality.

In [ ]:
low_dim = dimension_means.index[0]
high_dim = dimension_means.index[-1]
low_domain = domain_stats.iloc[0]
high_domain = domain_stats.iloc[-1]

summary = f"""### Synthetic benchmark correctness — findings

- **Overall task-weighted correctness:** **{overall:.3f} / 1.000**, computed across **{num_tasks} tasks** in **{num_domains} domains**.
- **Strongest dimension:** **{high_dim}** ({dimension_means.iloc[-1]:.3f}); **weakest dimension:** **{low_dim}** ({dimension_means.iloc[0]:.3f}).
- **Highest domain mean:** **{high_domain['domain_label']}** ({high_domain['mean_correctness']:.3f}; n={int(high_domain['task_count'])}).
- **Lowest domain mean:** **{low_domain['domain_label']}** ({low_domain['mean_correctness']:.3f}; n={int(low_domain['task_count'])}).
- Of **{len(judgments)} dimension judgments**, **{supported_n}** received 1.0, **{uncertain_n}** received 0.5, and **{violated_n}** received 0.0.

**Interpretive caution:** A 0.5 signifies uncertain or insufficient evidence, not a verified incorrect task. A 0.0 reflects a judge-identified violation that should be checked against its cited task, tool, or policy evidence. These scores evaluate benchmark task correctness, **not agent success rates**.
"""
display(Markdown(summary))


---
### Notes for reproducibility

- **Input:** The seven `judge_details.json` files, with `task_scores.csv` and `domain_scores.csv` used to check structure and consistency.
- **Scoring:** `task_score = (T + O + P + F) / 4`; `overall = mean(task_score for all loaded tasks)`.
- **No re-scoring:** This notebook only visualizes already saved LLM results; it does not call the OpenAI API or change the underlying evaluations.
- **When new results arrive:** Replace/update the corresponding files and rerun all cells. The task count and plots will recalculate automatically.
